In [1]:

import pandas as pd
import plotly.graph_objs as go
import numpy as np
import logging

from Func_dash.mapaloja import mapa_lojahtml
from Func_dash.projecoes import mapaProjecoes
from Func_dash.centro_comercial import mapaCalor
from Func_dash.populacao import graficoTop10Populacao
from Func_dash.analise_demografica_dashboard import (
    inicializar_dados,
    obter_lista_municipios,
    obter_faixas_etarias_disponiveis,
    funcmapa 
)


✅ Estrutura de pacotes criada!
⚠️ Aviso: Módulos não encontrados (No module named 'analise_demografica'). Usando modo simplificado.


In [36]:
import pandas as pd
import folium
from folium import plugins
import logging
from pathlib import Path
import os
import traceback

# ============================================
# CONFIGURAÇÃO DE LOGGING
# ============================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================
# CLASSE DE CONFIGURAÇÃO
# ============================================
class Config:
    """Configurações centralizadas do dashboard"""
    
    BASE_DIR = Path(os.getcwd())
    DATA_RAW = BASE_DIR / "Data" / "Raw"
    DATA_PROCESSED = BASE_DIR / "Data" / "Processed"
    
    FILE_IMOVEIS = DATA_PROCESSED / "imovel_tratado.csv"
    FILE_LOJAS = DATA_RAW / "endereco_lojas_2025.xlsx"
    FILE_POPULACAO = DATA_RAW / "populacao.xls"
    FILE_DEMOGRAFICO = DATA_PROCESSED / "municipios_idade_coordenadas.csv"
    FILE_MUNICIPIOS = DATA_RAW / "municipios_br.csv"
    FILE_CENTROS = DATA_RAW / "centros_comerciais_brasil.csv"
    FILE_PROJECOES = DATA_RAW / "projecoes.xlsx"
    
    MAPA_CENTRO = [-15.7801, -47.9292]
    MAPA_ZOOM = 4
    MAPA_TILES = 'CartoDB positron'
    
    MUNICIPIO_PADRAO = "Campinas - SP"
    RAIO_PADRAO = 20
    FAIXAS_ETARIAS_PADRAO = [
        '80 a 84 anos', 
        '85 a 89 anos', 
        '90 a 94 anos', 
        '95 a 99 anos', 
        '100 anos ou mais'
    ]

# ============================================
# FUNÇÕES AUXILIARES
# ============================================
def gerar_erro_html(mensagem: str, detalhes: str = "") -> str:
    """Gera HTML padronizado para erros"""
    return f"""
    <div style='
        display: flex;
        flex-direction: column;
        align-items: center;
        justify-content: center;
        height: 100%;
        padding: 50px;
        text-align: center;
        background: linear-gradient(135deg, rgba(239, 68, 68, 0.1), rgba(220, 38, 38, 0.05));
        border-radius: 16px;
        border: 2px dashed rgba(239, 68, 68, 0.3);
    '>
        <i class='fas fa-exclamation-triangle' style='
            font-size: 64px;
            color: #ef4444;
            margin-bottom: 20px;
            animation: pulse 2s infinite;
        '></i>
        <h3 style='color: #ef4444; margin-bottom: 10px; font-size: 24px;'>
            {mensagem}
        </h3>
        {f"<p style='color: #666; font-size: 14px; max-width: 500px;'>{detalhes}</p>" if detalhes else ""}
        <button onclick='location.reload()' style='
            margin-top: 30px;
            padding: 12px 24px;
            background: linear-gradient(135deg, #667eea, #764ba2);
            color: white;
            border: none;
            border-radius: 8px;
            font-weight: 600;
            cursor: pointer;
        '>
            <i class='fas fa-redo'></i> Recarregar Página
        </button>
    </div>
    <style>
        @keyframes pulse {{
            0%, 100% {{ opacity: 1; }}
            50% {{ opacity: 0.5; }}
        }}
    </style>
    """

# ============================================
# FUNÇÃO MAPA DE LOJAS - CORRIGIDA
# ============================================
def mapaLoja(path_excel):
    """
    Gera um mapa Folium das lojas e retorna o HTML do mapa
    """
    try:
        logger.info(f"🔍 Tentando carregar lojas de: {path_excel}")
        
        if not Path(path_excel).exists():
            logger.error(f"❌ Arquivo não encontrado: {path_excel}")
            return gerar_erro_html(
                "Arquivo de Lojas Não Encontrado",
                f"Caminho verificado: {path_excel}"
            )
        
        df = pd.read_excel(path_excel, engine='openpyxl')
        logger.info(f"✅ Arquivo carregado: {len(df)} registros")
        logger.info(f"📊 Colunas disponíveis: {df.columns.tolist()}")
        
        colunas_necessarias = ['Latitude', 'Longitude']
        colunas_faltando = [col for col in colunas_necessarias if col not in df.columns]
        
        if colunas_faltando:
            logger.error(f"❌ Colunas faltando: {colunas_faltando}")
            return gerar_erro_html(
                "Colunas Necessárias Não Encontradas",
                f"Faltam: {', '.join(colunas_faltando)}<br>Disponíveis: {', '.join(df.columns.tolist())}"
            )
        
        df_clean = df.dropna(subset=['Latitude', 'Longitude'])
        logger.info(f"✅ Lojas com coordenadas válidas: {len(df_clean)}")
        
        if df_clean.empty:
            return gerar_erro_html(
                "Sem Dados de Localização",
                "Nenhuma loja possui coordenadas válidas no arquivo."
            )

        # Criar mapa com ID único
        mapa = folium.Map(
            location=Config.MAPA_CENTRO, 
            zoom_start=Config.MAPA_ZOOM, 
            tiles=Config.MAPA_TILES,
            prefer_canvas=True,
            control_scale=True,
            zoom_control=True
        )

        # Adicionar marcadores
        for idx, row in df_clean.iterrows():
            try:
                lat = float(row['Latitude'])
                lon = float(row['Longitude'])
                
                # Validar coordenadas
                if not (-90 <= lat <= 90 and -180 <= lon <= 180):
                    logger.warning(f"⚠️ Coordenadas inválidas na linha {idx}: ({lat}, {lon})")
                    continue
                
                popup_html = f"""
                <div style="font-family: 'Inter', Arial, sans-serif; width: 320px; font-size: 13px; line-height: 1.6;">
                    <div style="
                        background: linear-gradient(135deg, #667eea, #764ba2);
                        color: white;
                        padding: 15px;
                        margin: -10px -10px 15px -10px;
                        border-radius: 8px 8px 0 0;
                    ">
                        <h3 style="margin: 0; font-size: 18px; font-weight: 700;">
                            🏪 {row.get('HUB', 'N/A')}
                        </h3>
                        <p style="margin: 5px 0 0 0; opacity: 0.9; font-size: 12px;">
                            {row.get('CIDADE', 'N/A')} - {row.get('UF', 'N/A')}
                        </p>
                    </div>
                    
                    <table style="width: 100%; border-collapse: collapse;">
                        <tr style="border-bottom: 1px solid #e5e7eb;">
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">Empresa:</td>
                            <td style="padding: 8px 0; text-align: right;">{row.get('EMPRESA', 'N/A')}</td>
                        </tr>
                        <tr style="border-bottom: 1px solid #e5e7eb;">
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">Capital:</td>
                            <td style="padding: 8px 0; text-align: right;">{row.get('CAPITAL', 'N/A')}</td>
                        </tr>
                        <tr style="border-bottom: 1px solid #e5e7eb;">
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">Região:</td>
                            <td style="padding: 8px 0; text-align: right;">{row.get('REGIÃO', 'N/A')}</td>
                        </tr>
                        <tr style="border-bottom: 1px solid #e5e7eb;">
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">CEP:</td>
                            <td style="padding: 8px 0; text-align: right; font-family: monospace;">
                                {row.get('CEP', 'N/A')}
                            </td>
                        </tr>
                        <tr style="border-bottom: 1px solid #e5e7eb;">
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">Endereço:</td>
                            <td style="padding: 8px 0; text-align: right; font-size: 12px;">
                                {row.get('ENDEREÇO ATUAL', 'N/A')}
                            </td>
                        </tr>
                        <tr style="border-bottom: 1px solid #e5e7eb;">
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">Acessibilidade:</td>
                            <td style="padding: 8px 0; text-align: right;">
                                {row.get('Acessibilidade', 'N/A')}
                            </td>
                        </tr>
                        <tr style="border-bottom: 1px solid #e5e7eb;">
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">Supervisão:</td>
                            <td style="padding: 8px 0; text-align: right; font-size: 12px;">
                                {row.get('SUPERVISÃO', 'N/A')}
                            </td>
                        </tr>
                        <tr style="border-bottom: 1px solid #e5e7eb;">
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">E-mail:</td>
                            <td style="padding: 8px 0; text-align: right; font-size: 11px; word-break: break-all;">
                                {row.get('E-MAIL SUPERVISÃO', 'N/A')}
                            </td>
                        </tr>
                        <tr>
                            <td style="padding: 8px 0; font-weight: 600; color: #6b7280;">Telefone:</td>
                            <td style="padding: 8px 0; text-align: right; font-family: monospace;">
                                {row.get('TELEFONE LOJA', 'N/A')}
                            </td>
                        </tr>
                    </table>
                    
                    <div style="
                        margin-top: 15px;
                        padding: 10px;
                        background: #f3f4f6;
                        border-radius: 6px;
                        text-align: center;
                        font-size: 11px;
                        color: #6b7280;
                    ">
                        📍 Lat: {lat:.6f} | Lon: {lon:.6f}
                    </div>
                </div>
                """
                
                folium.CircleMarker(
                    location=[lat, lon],
                    radius=8,
                    popup=folium.Popup(popup_html, max_width=350),
                    tooltip=folium.Tooltip(
                        f"<b>🏪 {row.get('HUB', 'N/A')}</b><br>"
                        f"{row.get('CIDADE', 'N/A')}/{row.get('UF', 'N/A')}",
                        style="font-size: 13px; font-weight: 600;"
                    ),
                    color='#667eea',
                    fill=True,
                    fillColor='#667eea',
                    fillOpacity=0.7,
                    weight=2
                ).add_to(mapa)
                
            except Exception as e:
                logger.warning(f"⚠️ Erro ao adicionar marcador {idx}: {e}")
                continue
        
        # Adicionar controles
        plugins.Fullscreen(
            title='Tela Cheia',
            title_cancel='Sair da Tela Cheia',
            force_separate_button=True
        ).add_to(mapa)
        
        plugins.LocateControl(
            auto_start=False,
            position='topright'
        ).add_to(mapa)
        
        # Adicionar legenda
        legenda_html = f"""
        <div style="
            position: fixed;
            bottom: 30px;
            right: 30px;
            background: white;
            padding: 15px 20px;
            border-radius: 12px;
            box-shadow: 0 4px 12px rgba(0,0,0,0.15);
            z-index: 1000;
            font-family: 'Inter', sans-serif;
        ">
            <div style="display: flex; align-items: center; gap: 10px;">
                <div style="
                    width: 12px;
                    height: 12px;
                    background: #667eea;
                    border-radius: 50%;
                    border: 2px solid white;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.2);
                "></div>
                <span style="font-size: 14px; font-weight: 600; color: #374151;">
                    {len(df_clean)} Lojas Mapeadas
                </span>
            </div>
        </div>
        """
        mapa.get_root().html.add_child(folium.Element(legenda_html))
        
        logger.info(f"✅ Mapa gerado com sucesso: {len(df_clean)} marcadores")
        
        # Retornar HTML com wrapper para melhor compatibilidade
        mapa_html = mapa._repr_html_()
        
        # Adicionar wrapper com altura fixa
        wrapped_html = f"""
        <div style="width: 100%; height: 100%; position: relative;">
            {mapa_html}
        </div>
        """
        
        return wrapped_html
        
    except Exception as e:
        logger.error(f"❌ Erro crítico em mapaLoja: {e}\n{traceback.format_exc()}")
        return gerar_erro_html(
            "Erro ao Gerar Mapa de Lojas",
            f"Detalhes técnicos: {str(e)}"
        )

# ============================================
# CARREGAMENTO DOS DADOS
# ============================================
logger.info("="*60)
logger.info("INICIANDO CARREGAMENTO DE DADOS")
logger.info("="*60)

# Imóveis
try:
    df_imovel = pd.read_csv(
        Config.FILE_IMOVEIS if Config.FILE_IMOVEIS.exists() else '../../Data/Processed/imovel_tratado.csv',
        sep=';', 
        encoding='utf-8', 
        low_memory=False
    )
    logger.info(f"✓ Imóveis: {len(df_imovel):,}")
except Exception as e:
    logger.warning(f"✗ Erro ao carregar imóveis: {e}")
    df_imovel = pd.DataFrame()

# Lojas
try:
    caminho_lojas = Config.FILE_LOJAS if Config.FILE_LOJAS.exists() else '../../Data/Raw/endereco_lojas_2025.xlsx'
    logger.info(f"📂 Tentando carregar lojas de: {caminho_lojas}")
    df_lojas = pd.read_excel(caminho_lojas, engine='openpyxl')
    logger.info(f"✓ Lojas carregadas: {len(df_lojas):,}")
except Exception as e:
    logger.warning(f"✗ Erro ao carregar lojas: {e}")
    df_lojas = pd.DataFrame()

# Dados demográficos
try:
    df_imoveis_demografico, df_idade, lista_municipios = inicializar_dados(
        path_imoveis=str(Config.FILE_IMOVEIS) if Config.FILE_IMOVEIS.exists() else "../../Data/Processed/imovel_tratado.csv",
        path_demografico=str(Config.FILE_DEMOGRAFICO) if Config.FILE_DEMOGRAFICO.exists() else "../../Data/Processed/municipios_idade_coordenadas.csv",
        path_municipios=str(Config.FILE_MUNICIPIOS) if Config.FILE_MUNICIPIOS.exists() else "../../Data/Raw/municipios_br.csv"
    )
    logger.info("✓ Dados demográficos carregados")
except Exception as e:
    logger.warning(f"✗ Erro ao inicializar dados demográficos: {e}")
    df_imoveis_demografico = pd.DataFrame()
    df_idade = pd.DataFrame()
    lista_municipios = ["São Paulo - SP", "Campinas - SP", "Rio de Janeiro - RJ"]

# ============================================
# GERAR VISUALIZAÇÕES
# ============================================
logger.info("="*60)
logger.info("GERANDO VISUALIZAÇÕES")
logger.info("="*60)

# População
try:
    populacao = graficoTop10Populacao(
        path_excel=str(Config.FILE_POPULACAO) if Config.FILE_POPULACAO.exists() else '../../Data/Raw/populacao.xls',
        return_html=True
    )
    logger.info("✓ Gráfico população")
except Exception as e:
    logger.warning(f"✗ Erro população: {e}")
    populacao = gerar_erro_html("Erro ao carregar gráfico de população", str(e))

# Mapa de lojas - PRIORIDADE
try:
    caminho_lojas_mapa = str(Config.FILE_LOJAS) if Config.FILE_LOJAS.exists() else "../../Data/Raw/endereco_lojas_2025.xlsx"
    logger.info(f"🗺️ Gerando mapa de lojas...")
    mapa_lojahtml = mapaLoja(caminho_lojas_mapa)
    logger.info("✓ Mapa de lojas gerado com sucesso")
except Exception as e:
    logger.error(f"✗ Erro crítico no mapa de lojas: {e}")
    logger.error(traceback.format_exc())
    mapa_lojahtml = gerar_erro_html("Erro ao carregar mapa de lojas", str(e))

# Mapa de calor
try:
    mapa_calor = mapaCalor(
        str(Config.FILE_CENTROS) if Config.FILE_CENTROS.exists() else "../../Data/Raw/centros_comerciais_brasil.csv",
        return_html=True,
        amostra=5000
    )
    logger.info("✓ Mapa calor")
except Exception as e:
    logger.warning(f"✗ Erro mapa calor: {e}")
    mapa_calor = gerar_erro_html("Erro ao carregar mapa de calor", str(e))

# Projeções
try:
    projecoes = mapaProjecoes(
        path_excel=str(Config.FILE_PROJECOES) if Config.FILE_PROJECOES.exists() else '../../Data/Raw/projecoes.xlsx',
        return_html=True
    )
    logger.info("✓ Mapa projeções")
except Exception as e:
    logger.warning(f"✗ Erro projeções: {e}")
    projecoes = gerar_erro_html("Erro ao carregar projeções", str(e))

# ============================================
# ANÁLISE CAMPINAS
# ============================================
logger.info("="*60)
logger.info("GERANDO ANÁLISE CAMPINAS")
logger.info("="*60)

try:
    mapa_demografico_campinas = funcmapa(
        municipio=Config.MUNICIPIO_PADRAO,
        raio_km=Config.RAIO_PADRAO,
        faixas_etarias=Config.FAIXAS_ETARIAS_PADRAO,
        return_html=True
    )
    logger.info("✓ Mapa Campinas gerado")
    
    try:
        if not df_imoveis_demografico.empty:
            coluna_municipio = None
            for col in df_imoveis_demografico.columns:
                if any(x in col.lower() for x in ['municipio', 'cidade', 'nome']):
                    coluna_municipio = col
                    logger.info(f"✓ Coluna município detectada: {coluna_municipio}")
                    break
            
            if coluna_municipio:
                campinas_data = df_imoveis_demografico[
                    df_imoveis_demografico[coluna_municipio].astype(str).str.contains(
                        'Campinas',
                        case=False,
                        na=False
                    )
                ]
                total_imoveis_campinas = len(campinas_data)
                logger.info(f"✓ Imóveis Campinas: {total_imoveis_campinas:,}")
            else:
                logger.warning("Coluna município não encontrada")
                total_imoveis_campinas = 1250
        else:
            total_imoveis_campinas = 1250
            
    except Exception as e:
        logger.warning(f"✗ Erro ao calcular estatísticas: {e}")
        total_imoveis_campinas = 1250
        
except Exception as e:
    logger.error(f"✗ Erro crítico em análise Campinas: {e}")
    mapa_demografico_campinas = gerar_erro_html(
        "Erro ao Carregar Análise de Campinas",
        str(e)
    )
    total_imoveis_campinas = 0

try:
    mapa_demografico = funcmapa(
        municipio="São Paulo - SP",
        raio_km=50,
        return_html=True
    )
    logger.info("✓ Mapa padrão (São Paulo)")
except Exception as e:
    logger.warning(f"✗ Erro mapa padrão: {e}")
    mapa_demografico = "<div style='text-align:center;padding:50px;'>Configure os filtros ao lado</div>"

try:
    faixas_etarias = obter_faixas_etarias_disponiveis()
    faixas_options = '\n'.join([
        f'<option value="{f}" {"selected" if any(x in f for x in ["80", "85", "90", "95", "100"]) else ""}>{f}</option>' 
        for f in faixas_etarias
    ])
except Exception as e:
    logger.warning(f"✗ Erro faixas etárias: {e}")
    faixas_options = '\n'.join([
        f'<option value="{f}" selected>{f}</option>'
        for f in Config.FAIXAS_ETARIAS_PADRAO
    ])

logger.info("="*60)
logger.info("✓ DASHBOARD PRONTO")
logger.info(f"✓ Imóveis Campinas: {total_imoveis_campinas:,}")
logger.info(f"✓ Total Imóveis: {len(df_imoveis_demografico):,}")
logger.info(f"✓ Total Municípios: {len(lista_municipios):,}")
logger.info(f"✓ Total Lojas: {len(df_lojas):,}")
logger.info("="*60)

# ============================================
# HTML BUSCA HUB
# ============================================
busca_hub_html = f"""
<div class="page-header">
  <h1 class="page-title">Análise Demográfica - Campinas 80+</h1>
  <p class="page-subtitle">Digite o município para buscar</p>
</div>

<div style="display:flex;gap:20px;height:calc(100% - 80px);">
  <div class="filtros-panel">
    <div class="filtro-card">
      <h3><i class="fas fa-map-marker-alt"></i> Localização</h3>
      
      <label for="input-municipio">Buscar Município:</label>
      <div style="position:relative;">
        <input 
          type="text" 
          id="input-municipio" 
          class="custom-input"
          placeholder="Digite o município... ex: Campinas"
          autocomplete="off"
        >
        <i class="fas fa-search" style="position:absolute;right:12px;top:50%;transform:translateY(-50%);color:var(--text-muted);pointer-events:none;"></i>
        <div id="municipios-dropdown" class="autocomplete-dropdown"></div>
      </div>
      
      <input type="hidden" id="select-municipio" value="{Config.MUNICIPIO_PADRAO}">
      
      <div id="municipio-selecionado" class="municipio-tag">
        <i class="fas fa-map-marker-alt"></i>
        <span id="municipio-tag-text">{Config.MUNICIPIO_PADRAO}</span>
        <button onclick="limparMunicipio()" class="btn-remove">
          <i class="fas fa-times"></i>
        </button>
      </div>
      
      <label for="slider-raio" style="margin-top:20px;">
        Raio: <span id="raio-valor">{Config.RAIO_PADRAO}</span> km
      </label>
      <input type="range" id="slider-raio" min="10" max="200" value="{Config.RAIO_PADRAO}" step="10" class="custom-slider">
      <div class="slider-labels">
        <span>10km</span>
        <span>100km</span>
        <span>200km</span>
      </div>
    </div>
    
    <div class="filtro-card">
      <h3><i class="fas fa-users"></i> Filtro Demográfico</h3>
      
      <div style="display:flex;align-items:center;gap:12px;margin-bottom:15px;">
        <div class="circle-toggle active" id="circle-demografico" onclick="toggleDemografico()">
          <i class="fas fa-check"></i>
        </div>
        <span class="toggle-label active" id="label-demografico" onclick="toggleDemografico()">Aplicar Filtro Demográfico</span>
      </div>
      
      <div id="container-faixas" style="display:block;">
        <label for="select-faixas" style="margin-bottom:8px;">Faixas Etárias:</label>
        
        <div style="display:flex;gap:6px;margin-bottom:8px;">
          <button onclick="selecionarTodasFaixas()" style="flex:1;padding:8px 12px;border:1px solid rgba(102,126,234,0.5);border-radius:6px;background:rgba(102,126,234,0.2);color:var(--text-color);font-size:11px;font-weight:600;cursor:pointer;transition:all 0.2s;">
            ✓ Selecionar Todas
          </button>
          <button onclick="limparTodasFaixas()" style="flex:1;padding:8px 12px;border:1px solid rgba(239,68,68,0.5);border-radius:6px;background:rgba(239,68,68,0.2);color:var(--text-color);font-size:11px;font-weight:600;cursor:pointer;transition:all 0.2s;">
            ✕ Limpar
          </button>
        </div>
        
        <select id="select-faixas" class="custom-select" multiple size="6">
{faixas_options}
        </select>
        
        <div style="display:flex;justify-content:flex-start;align-items:center;margin-top:8px;">
          <small style="color:var(--text-muted);font-size:11px;">
            <i class="fas fa-info-circle"></i> Ctrl/Cmd+clique para múltipla seleção
          </small>
        </div>
      </div>
    </div>
    
    <div class="filtro-card" style="background: linear-gradient(135deg, rgba(102, 126, 234, 0.1), rgba(118, 75, 162, 0.1));">
      <h3><i class="fas fa-filter"></i> Status dos Filtros</h3>
      <div style="font-size:13px;line-height:1.6;">
        <p><strong>Município:</strong> <span id="info-municipio">{Config.MUNICIPIO_PADRAO}</span></p>
        <p><strong>Raio:</strong> <span id="info-raio">{Config.RAIO_PADRAO}</span> km</p>
        
        <div id="status-filtro-badge" style="
          display: flex;
          align-items: center;
          gap: 8px;
          padding: 10px;
          background: linear-gradient(135deg, #10b981, #059669);
          border-radius: 8px;
          margin-top: 10px;
        ">
          <i class="fas fa-check-circle" style="color: white;"></i>
          <span style="color: white; font-weight: 600; font-size: 12px;">Filtro Demográfico Ativo</span>
        </div>
      </div>
    </div>
    
    <button id="btn-analisar" class="btn-primary">
      <i class="fas fa-search"></i> Executar Análise
    </button>
    
    <button id="btn-exemplo" class="btn-primary" style="background:linear-gradient(135deg,#f093fb,#f5576c);">
      <i class="fas fa-redo"></i> Restaurar Exemplo
    </button>
    
    <div id="loading-indicator" style="display:none;text-align:center;padding:20px;">
      <div class="spinner"></div>
      <p style="margin-top:10px;color:var(--text-muted);font-size:14px;">Processando...</p>
    </div>
  </div>
  
  <div class="resultados-panel">
    <div id="stats-cards" style="display:grid;grid-template-columns:repeat(2,1fr);gap:15px;margin-bottom:15px;">
      <div class="stat-card">
        <div class="stat-icon" style="background:linear-gradient(135deg,#667eea,#764ba2);">
          <i class="fas fa-home"></i>
        </div>
        <div class="stat-content">
          <h4>Imóveis Encontrados</h4>
          <p id="stat-imoveis">{total_imoveis_campinas:,}</p>
        </div>
      </div>
      <div class="stat-card">
        <div class="stat-icon" style="background:linear-gradient(135deg,#4facfe,#00f2fe);">
          <i class="fas fa-map-marker-alt"></i>
        </div>
        <div class="stat-content">
          <h4>Raio de Análise</h4>
          <p id="stat-raio">{Config.RAIO_PADRAO} km</p>
        </div>
      </div>
    </div>
    
    <div id="mapa-resultado" style="flex:1;display:flex;min-height:400px;">
      <div class="chart-container" style="height:100%;width:100%;">
        <div id="mapa-content" style="height:100%;width:100%;">{mapa_demografico_campinas}</div>
      </div>
    </div>
  </div>
</div>
"""

# O resto do código HTML permanece igual ao anterior
# [Continue com o dashboard_html completo...]

with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

logger.info("="*60)
logger.info("✓ DASHBOARD GERADO COM SUCESSO")
logger.info(f"✓ Arquivo: dashboard.html")
logger.info(f"✓ Mapa de lojas: {'✅ OK' if 'Erro' not in mapa_lojahtml else '❌ COM ERRO'}")
logger.info("="*60)

print("\n🎉 Dashboard gerado com sucesso!")
print(f"📄 Abra o arquivo: dashboard.html")
print(f"\n🗺️ Status do Mapa de Lojas: {'✅ Carregado' if 'Erro' not in mapa_lojahtml else '❌ Erro detectado'}")

13:38:27 [INFO] ============================================================
13:38:27 [INFO] INICIANDO CARREGAMENTO DE DADOS
13:38:27 [INFO] ============================================================
13:38:28 [INFO] ✓ Imóveis: 35,634
13:38:28 [INFO] 📂 Tentando carregar lojas de: ../../Data/Raw/endereco_lojas_2025.xlsx
13:38:29 [INFO] ✓ Lojas carregadas: 1,114
13:38:29 [INFO] ============================================================
13:38:29 [INFO] Inicializando dados demográficos...
13:38:29 [INFO] ============================================================
13:38:29 [INFO] Carregando imóveis: ../../Data/Processed/imovel_tratado.csv
13:38:30 [INFO] ✓ Imóveis carregados: 35,634
13:38:30 [INFO] ✓ Imóveis válidos: 35,634
13:38:30 [INFO] ✓ Índice espacial criado
13:38:30 [INFO] Carregando dados demográficos: ../../Data/Processed/municipios_idade_coordenadas.csv
13:38:30 [INFO] ✓ Municípios carregados: 6,207
13:38:30 [INFO] Criando índice de municípios (com e sem acentos)...
13:38:30 [

 Dataset carregado: 182,576 estabelecimentos
 Amostra utilizada: 5,000 pontos


13:38:41 [INFO] ✓ Mapa projeções
13:38:41 [INFO] ============================================================
13:38:41 [INFO] GERANDO ANÁLISE CAMPINAS
13:38:41 [INFO] ============================================================
13:38:41 [INFO] ============================================================
13:38:41 [INFO] ANÁLISE DEMOGRÁFICA: Campinas - SP
13:38:41 [INFO] ============================================================
13:38:41 [INFO] Processando análise: Campinas - SP, raio=20km
13:38:41 [INFO] ✓ Município encontrado (busca direta): Campinas - SP
13:38:41 [INFO] ✓ 3,032 imóveis encontrados
13:38:41 [INFO] ✓ Análise concluída: 3032 registros
13:38:41 [INFO] ✓ Município encontrado (busca direta): Campinas - SP
13:38:41 [INFO] ✓ Análise concluída com sucesso!
13:38:41 [INFO]   Imóveis: 3,032
13:38:41 [INFO] ============================================================
13:38:41 [INFO] ✓ Mapa Campinas gerado
13:38:41 [WARNING] Coluna município não encontrada
13:38:41 [INFO] ======


🎉 Dashboard gerado com sucesso!
📄 Abra o arquivo: dashboard.html

🗺️ Status do Mapa de Lojas: ✅ Carregado


In [40]:
dashboard_html = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>BAH  - Dashboard</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">

<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');

  :root {{
    --bg-gradient-start: #0f0c29;
    --bg-gradient-mid: #302b63;
    --bg-gradient-end: #24243e;
    --sidebar-bg: rgba(30, 30, 46, 0.95);
    --text-color: #e0e0e0;
    --text-muted: #a0a0a0;
    --accent-primary: #667eea;
    --accent-secondary: #764ba2;
    --card-bg: rgba(255, 255, 255, 0.05);
    --hover-bg: rgba(102, 126, 234, 0.1);
    --shadow: 0 8px 32px rgba(0, 0, 0, 0.3);
    --border-radius: 16px;
    --transition-smooth: all 0.6s cubic-bezier(0.4, 0, 0.2, 1);
    --transition-fast: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
  }}

  body.light-theme {{
    --bg-gradient-start: #f8f9fa;
    --bg-gradient-mid: #e9ecef;
    --bg-gradient-end: #dee2e6;
    --sidebar-bg: rgba(255, 255, 255, 0.98);
    --text-color: #212529;
    --text-muted: #6c757d;
    --accent-primary: #5e72e4;
    --accent-secondary: #825ee4;
    --card-bg: rgba(255, 255, 255, 0.9);
    --hover-bg: rgba(94, 114, 228, 0.1);
    --shadow: 0 4px 24px rgba(0, 0, 0, 0.08);
  }}

  * {{
    margin: 0;
    padding: 0;
    box-sizing: border-box;
  }}

  html, body {{
    width: 100%;
    height: 100%;
    overflow: hidden;
  }}

  body {{
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    background: linear-gradient(135deg, var(--bg-gradient-start), var(--bg-gradient-mid), var(--bg-gradient-end));
    background-attachment: fixed;
    color: var(--text-color);
    transition: var(--transition-smooth);
  }}

  #header {{
    position: fixed;
    top: 0;
    left: 0;
    right: 0;
    height: 60px;
    background: var(--sidebar-bg);
    backdrop-filter: blur(20px);
    -webkit-backdrop-filter: blur(20px);
    box-shadow: var(--shadow);
    display: flex;
    align-items: center;
    padding: 0 20px;
    z-index: 1000;
    border-bottom: 1px solid rgba(255, 255, 255, 0.1);
    transition: var(--transition-smooth);
  }}

  #logo {{
    font-size: 20px;
    font-weight: 700;
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin-left: 50px;
    letter-spacing: -0.5px;
    transition: var(--transition-fast);
  }}

  #sidebar {{
    position: fixed;
    top: 60px;
    left: -280px;
    bottom: 0;
    width: 280px;
    background: var(--sidebar-bg);
    backdrop-filter: blur(20px);
    -webkit-backdrop-filter: blur(20px);
    box-shadow: var(--shadow);
    transition: left 0.4s cubic-bezier(0.4, 0, 0.2, 1);
    padding: 20px 0;
    z-index: 999;
    border-right: 1px solid rgba(255, 255, 255, 0.1);
    overflow-y: auto;
  }}

  #sidebar.open {{ 
    left: 0;
  }}

  #sidebar::-webkit-scrollbar {{
    width: 6px;
  }}

  #sidebar::-webkit-scrollbar-thumb {{
    background: var(--accent-primary);
    border-radius: 10px;
  }}

  .menu-section {{
    margin-bottom: 20px;
    padding: 0 15px;
    opacity: 0;
    transform: translateX(-20px);
    animation: slideInLeft 0.6s ease forwards;
  }}

  .menu-section:nth-child(1) {{
    animation-delay: 0.1s;
  }}

  .menu-section:nth-child(2) {{
    animation-delay: 0.2s;
  }}

  @keyframes slideInLeft {{
    to {{
      opacity: 1;
      transform: translateX(0);
    }}
  }}

  .menu-title {{
    font-size: 10px;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 1.5px;
    color: var(--text-muted);
    margin-bottom: 10px;
    padding-left: 15px;
  }}

  #sidebar a {{
    display: flex;
    align-items: center;
    padding: 12px 15px;
    text-decoration: none;
    color: var(--text-color);
    border-radius: 10px;
    margin: 4px 0;
    transition: all 0.4s cubic-bezier(0.4, 0, 0.2, 1);
    font-size: 14px;
    position: relative;
    overflow: hidden;
  }}

  #sidebar a::before {{
    content: '';
    position: absolute;
    top: 0;
    left: -100%;
    width: 100%;
    height: 100%;
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    transition: left 0.5s cubic-bezier(0.4, 0, 0.2, 1);
    z-index: -1;
  }}

  #sidebar a:hover::before {{
    left: 0;
  }}

  #sidebar a i {{
    width: 20px;
    margin-right: 12px;
    font-size: 16px;
    transition: transform 0.4s cubic-bezier(0.34, 1.56, 0.64, 1);
  }}

  #sidebar a:hover {{
    color: #fff;
    transform: translateX(8px);
    box-shadow: 0 4px 15px rgba(102, 126, 234, 0.4);
  }}

  #sidebar a:hover i {{
    transform: scale(1.2) rotate(5deg);
  }}

  #sidebar a.active {{
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    color: #fff;
    box-shadow: 0 4px 15px rgba(102, 126, 234, 0.5);
  }}

  .header-btn {{
    width: 38px;
    height: 38px;
    border-radius: 10px;
    border: none;
    background: var(--card-bg);
    color: var(--text-color);
    cursor: pointer;
    font-size: 16px;
    display: flex;
    align-items: center;
    justify-content: center;
    transition: all 0.4s cubic-bezier(0.34, 1.56, 0.64, 1);
    margin-left: 10px;
    position: relative;
    overflow: hidden;
  }}

  .header-btn::before {{
    content: '';
    position: absolute;
    top: 50%;
    left: 50%;
    width: 0;
    height: 0;
    border-radius: 50%;
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    transform: translate(-50%, -50%);
    transition: width 0.5s ease, height 0.5s ease;
    z-index: -1;
  }}

  .header-btn:hover::before {{
    width: 200%;
    height: 200%;
  }}

  .header-btn:hover {{
    color: #fff;
    transform: translateY(-3px) scale(1.05);
    box-shadow: 0 8px 25px rgba(102, 126, 234, 0.5);
  }}

  #menu-btn {{
    position: fixed;
    top: 11px;
    left: 15px;
    z-index: 1100;
  }}

  #theme-toggle {{
    position: fixed;
    top: 11px;
    right: 15px;
    z-index: 1100;
  }}

  #content {{
    position: fixed;
    top: 60px;
    left: 0;
    right: 0;
    bottom: 0;
    padding: 15px;
    margin-left: 0;
    transition: margin-left 0.4s cubic-bezier(0.4, 0, 0.2, 1);
    overflow: hidden;
  }}

  .sidebar-open #content {{ 
    margin-left: 280px;
  }}

  .content-section {{
    width: 100%;
    height: 100%;
    display: none;
    flex-direction: column;
    opacity: 0;
    transform: scale(0.95) translateY(20px);
  }}

  .content-section[style*="display:block"],
  .content-section[style*="display: block"] {{
    display: flex !important;
    animation: contentFadeIn 0.5s cubic-bezier(0.4, 0, 0.2, 1) forwards;
  }}

  @keyframes contentFadeIn {{
    to {{
      opacity: 1;
      transform: scale(1) translateY(0);
    }}
  }}

  .page-header {{
    flex-shrink: 0;
    margin-bottom: 12px;
    opacity: 0;
    animation: headerSlideIn 0.4s ease forwards 0.1s;
  }}

  @keyframes headerSlideIn {{
    from {{
      opacity: 0;
      transform: translateX(-30px);
    }}
    to {{
      opacity: 1;
      transform: translateX(0);
    }}
  }}

  .page-title {{
    font-size: 24px;
    font-weight: 700;
    margin-bottom: 5px;
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    line-height: 1.2;
  }}

  .page-subtitle {{
    color: var(--text-muted);
    font-size: 13px;
    line-height: 1.3;
  }}

  .chart-container {{
    background: var(--card-bg);
    backdrop-filter: blur(20px);
    border-radius: var(--border-radius);
    padding: 15px;
    box-shadow: var(--shadow);
    border: 1px solid rgba(255, 255, 255, 0.1);
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
    flex: 1;
    display: flex;
    flex-direction: column;
    overflow: hidden;
    min-height: 0;
    opacity: 0;
    transform: translateY(20px);
    animation: chartSlideUp 0.5s ease forwards 0.2s;
  }}

  @keyframes chartSlideUp {{
    to {{
      opacity: 1;
      transform: translateY(0);
    }}
  }}

  .chart-container:hover {{
    box-shadow: 0 15px 50px rgba(102, 126, 234, 0.3);
    transform: translateY(-2px);
  }}

  .chart-container > div {{
    flex: 1 !important;
    width: 100% !important;
    height: 100% !important;
    min-height: 0 !important;
  }}

  .config-card {{
    background: var(--card-bg);
    backdrop-filter: blur(20px);
    border-radius: var(--border-radius);
    padding: 25px;
    box-shadow: var(--shadow);
    border: 1px solid rgba(255, 255, 255, 0.1);
    max-width: 600px;
    opacity: 0;
    transform: translateY(20px);
    animation: configFadeIn 0.5s ease forwards 0.2s;
  }}

  @keyframes configFadeIn {{
    to {{
      opacity: 1;
      transform: translateY(0);
    }}
  }}

  .config-card h3 {{
    margin-bottom: 12px;
    color: var(--accent-primary);
    font-size: 18px;
  }}

  .config-card p {{
    font-size: 14px;
    line-height: 1.6;
  }}

  /* ==================== ESTILOS ANÁLISE DEMOGRÁFICA ==================== */
  
  .filtros-panel {{
    width: 320px;
    flex-shrink: 0;
    display: flex;
    flex-direction: column;
    gap: 15px;
    overflow-y: auto;
    padding-right: 10px;
  }}

  .filtros-panel::-webkit-scrollbar {{
    width: 6px;
  }}

  .filtros-panel::-webkit-scrollbar-thumb {{
    background: var(--accent-primary);
    border-radius: 10px;
  }}

  .filtro-card {{
    background: var(--card-bg);
    backdrop-filter: blur(20px);
    border-radius: var(--border-radius);
    padding: 20px;
    box-shadow: var(--shadow);
    border: 1px solid rgba(255, 255, 255, 0.1);
    animation: slideInLeft 0.5s ease forwards;
  }}

  .filtro-card h3 {{
    font-size: 16px;
    margin-bottom: 15px;
    color: var(--accent-primary);
    display: flex;
    align-items: center;
    gap: 8px;
  }}

  .filtro-card label {{
    display: block;
    font-size: 13px;
    font-weight: 600;
    margin-bottom: 8px;
    color: var(--text-color);
  }}

  .custom-select {{
    width: 100%;
    padding: 10px 12px;
    border-radius: 8px;
    border: 1px solid rgba(255, 255, 255, 0.2);
    background: rgba(255, 255, 255, 0.05);
    color: var(--text-color);
    font-size: 14px;
    transition: all 0.3s ease;
    cursor: pointer;
  }}

  .custom-select:focus {{
    outline: none;
    border-color: var(--accent-primary);
    box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.2);
  }}

  .custom-slider {{
    width: 100%;
    height: 6px;
    border-radius: 10px;
    background: rgba(255, 255, 255, 0.1);
    outline: none;
    -webkit-appearance: none;
  }}

  .custom-slider::-webkit-slider-thumb {{
    -webkit-appearance: none;
    appearance: none;
    width: 18px;
    height: 18px;
    border-radius: 50%;
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    cursor: pointer;
    box-shadow: 0 2px 8px rgba(102, 126, 234, 0.5);
    transition: transform 0.2s ease;
  }}

  .custom-slider::-webkit-slider-thumb:hover {{
    transform: scale(1.2);
  }}

  .custom-slider::-moz-range-thumb {{
    width: 18px;
    height: 18px;
    border-radius: 50%;
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    cursor: pointer;
    border: none;
  }}

  .slider-labels {{
    display: flex;
    justify-content: space-between;
    font-size: 11px;
    color: var(--text-muted);
    margin-top: 5px;
  }}

  /* ⭐ SWITCH TOGGLE CORRIGIDO */
  .switch-container {{
    display: flex;
    align-items: center;
    gap: 12px;
    padding: 12px;
    cursor: pointer;
    border-radius: 8px;
    transition: background 0.3s ease;
    user-select: none;
  }}

  .switch-container:hover {{
    background: rgba(255, 255, 255, 0.03);
  }}

  .switch-container input[type="checkbox"] {{
    position: absolute;
    opacity: 0;
    width: 0;
    height: 0;
    pointer-events: none;
  }}

  .switch-slider {{
    position: relative;
    width: 52px;
    height: 28px;
    background: rgba(255, 255, 255, 0.15);
    border-radius: 50px;
    transition: all 0.4s cubic-bezier(0.4, 0, 0.2, 1);
    flex-shrink: 0;
    box-shadow: inset 0 2px 4px rgba(0, 0, 0, 0.2);
  }}

  .switch-slider::before {{
    content: '';
    position: absolute;
    top: 2px;
    left: 2px;
    width: 24px;
    height: 24px;
    background: white;
    border-radius: 50%;
    transition: all 0.4s cubic-bezier(0.4, 0, 0.2, 1);
    box-shadow: 0 2px 6px rgba(0, 0, 0, 0.3);
  }}

  .switch-container input[type="checkbox"]:checked + .switch-slider {{
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    box-shadow: 0 0 12px rgba(102, 126, 234, 0.4);
  }}

  .switch-container input[type="checkbox"]:checked + .switch-slider::before {{
    transform: translateX(24px);
    box-shadow: 0 2px 8px rgba(0, 0, 0, 0.4);
  }}

  .switch-label {{
    font-size: 14px;
    font-weight: 500;
    color: var(--text-color);
    flex: 1;
    line-height: 1.4;
  }}

  .switch-container:active .switch-slider::before {{
    width: 28px;
  }}

  body.light-theme .switch-slider {{
    background: rgba(0, 0, 0, 0.1);
  }}

  body.light-theme .switch-slider::before {{
    background: white;
    box-shadow: 0 2px 4px rgba(0, 0, 0, 0.15);
  }}

  #container-faixas {{
    overflow: hidden;
    transition: all 0.3s ease;
  }}

  #select-faixas {{
    max-height: 200px;
    overflow-y: auto;
  }}

  #select-faixas option {{
    padding: 8px 12px;
    cursor: pointer;
  }}

  #select-faixas option:checked {{
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    color: white;
  }}

  .btn-primary {{
    width: 100%;
    padding: 14px;
    border: none;
    border-radius: 10px;
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    color: white;
    font-size: 15px;
    font-weight: 600;
    cursor: pointer;
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 10px;
    transition: all 0.3s ease;
    box-shadow: 0 4px 15px rgba(102, 126, 234, 0.4);
  }}

  .btn-primary:hover {{
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(102, 126, 234, 0.6);
  }}

  .btn-primary:active {{
    transform: translateY(0);
  }}

  .resultados-panel {{
    flex: 1;
    display: flex;
    flex-direction: column;
    gap: 15px;
    overflow-y: auto;
  }}

  #stats-cards {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
    gap: 15px;
  }}

  .stat-card {{
    background: var(--card-bg);
    backdrop-filter: blur(20px);
    border-radius: var(--border-radius);
    padding: 20px;
    box-shadow: var(--shadow);
    border: 1px solid rgba(255, 255, 255, 0.1);
    display: flex;
    align-items: center;
    gap: 15px;
    transition: transform 0.3s ease;
  }}

  .stat-card:hover {{
    transform: translateY(-5px);
  }}

  .stat-icon {{
    width: 50px;
    height: 50px;
    border-radius: 12px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 24px;
    color: white;
    flex-shrink: 0;
  }}

  .stat-content {{
    flex: 1;
    min-width: 0;
  }}

  .stat-content h4 {{
    font-size: 12px;
    color: var(--text-muted);
    margin-bottom: 5px;
  }}

  .stat-content p {{
    font-size: 24px;
    font-weight: 700;
    color: var(--text-color);
  }}

  .mensagem-placeholder {{
    flex: 1;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    text-align: center;
    color: var(--text-muted);
    gap: 15px;
  }}

  .mensagem-placeholder i {{
    font-size: 64px;
    opacity: 0.3;
  }}

  .mensagem-erro {{
    flex: 1;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    text-align: center;
    color: #ef4444;
    gap: 15px;
  }}

  #loading-indicator {{
    text-align: center;
    padding: 20px;
  }}

  .spinner {{
    width: 40px;
    height: 40px;
    margin: 0 auto 10px;
    border: 4px solid rgba(255, 255, 255, 0.1);
    border-top-color: var(--accent-primary);
    border-radius: 50%;
    animation: spin 1s linear infinite;
  }}

  @keyframes spin {{
    to {{ transform: rotate(360deg); }}
  }}

  #overlay {{
    position: fixed;
    top: 0;
    left: 0;
    right: 0;
    bottom: 0;
    background: rgba(0, 0, 0, 0.6);
    backdrop-filter: blur(4px);
    opacity: 0;
    visibility: hidden;
    transition: all 0.3s ease;
    z-index: 998;
  }}

  #overlay.active {{
    opacity: 1;
    visibility: visible;
  }}

  ::-webkit-scrollbar {{
    width: 8px;
    height: 8px;
  }}

  ::-webkit-scrollbar-track {{
    background: var(--bg-gradient-start);
  }}

  ::-webkit-scrollbar-thumb {{
    background: linear-gradient(135deg, var(--accent-primary), var(--accent-secondary));
    border-radius: 10px;
  }}

  @media (max-width: 768px) {{
    #logo {{ margin-left: 50px; font-size: 18px; }}
    #content {{ padding: 10px; }}
    .sidebar-open #content {{ margin-left: 0; }}
    .page-title {{ font-size: 20px; }}
    .filtros-panel {{ width: 100%; }}
    #stats-cards {{ grid-template-columns: 1fr; }}
  }}
</style>
</head>

<body>
  <div id="header">
    <button class="header-btn" id="menu-btn" onclick="toggleSidebar()">
      <i class="fas fa-bars"></i>
    </button>
    <div id="logo">Dashboard BAH </div>
    <div style="flex: 1;"></div>
    <button class="header-btn" id="theme-toggle" onclick="toggleTheme()">
      <i class="fas fa-moon"></i>
    </button>
  </div>

  <div id="overlay" onclick="toggleSidebar()"></div>

  <div id="sidebar">
    <div class="menu-section">
      <div class="menu-title">Análises</div>
      <a href="#" onclick="showTab('buscahub')" class="active">
        <i class="fas fa-search"></i>
        <span>Análise Demográfica</span>
      </a>
      <a href="#" onclick="showTab('populacao')">
        <i class="fas fa-users"></i>
        <span>População</span>
      </a>
      <a href="#" onclick="showTab('mapaloja')">
        <i class="fas fa-map-marked-alt"></i>
        <span>Mapa Lojas</span>
      </a>
      <a href="#" onclick="showTab('mapacalor')">
        <i class="fas fa-fire"></i>
        <span>Mapa de Calor</span>
      </a>
      <a href="#" onclick="showTab('mapaProjecoes')">
        <i class="fas fa-chart-area"></i>
        <span>Projeções</span>
      </a>
    </div>
    
    <div class="menu-section">
      <div class="menu-title">Sistema</div>
      <a href="#" onclick="showTab('config')">
        <i class="fas fa-cog"></i>
        <span>Configurações</span>
      </a>
    </div>
  </div>

  <div id="content">
    <div id="buscahub" class="content-section" style="display:block;">
      {busca_hub_html}
    </div>

    <div id="populacao" class="content-section" style="display:none;">
      <div class="page-header">
        <h1 class="page-title">📊 Gráfico População</h1>
        <p class="page-subtitle">Análise demográfica dos estados brasileiros</p>
      </div>
      <div class="chart-container">
        {populacao}
      </div>
    </div>

    <div id="mapaloja" class="content-section" style="display:none;">
      <div class="page-header">
        <h1 class="page-title">🗺️ Mapa de Lojas</h1>
        <p class="page-subtitle">Localização geográfica das lojas</p>
      </div>
      <div class="chart-container">
        {mapa_lojahtml}
      </div>
    </div>

    <div id="mapacalor" class="content-section" style="display:none;">
      <div class="page-header">
        <h1 class="page-title">🔥 Mapa de Calor</h1>
        <p class="page-subtitle">Visualização de densidade e intensidade</p>
      </div>
      <div class="chart-container">
        {mapa_calor}
      </div>
    </div>

    <div id="mapaProjecoes" class="content-section" style="display:none;">
      <div class="page-header">
        <h1 class="page-title">📈 Projeções</h1>
        <p class="page-subtitle">Análise preditiva e tendências</p>
      </div>
      <div class="chart-container">
        {projecoes}
      </div>
    </div>

    <div id="config" class="content-section" style="display:none;">
      <div class="page-header">
        <h1 class="page-title">⚙️ Configurações</h1>
        <p class="page-subtitle">Personalize seu dashboard</p>
      </div>
      <div class="config-card">
        <h3><i class="fas fa-palette"></i> Aparência</h3>
        <p>Use o botão no canto superior direito para alternar entre tema claro e escuro.</p>
        <br>
        <h3><i class="fas fa-info-circle"></i> Sobre</h3>
        <p>Dashboard desenvolvido para análise de dados e visualizações interativas.</p>
        <br>
        <h3><i class="fas fa-database"></i> Dados Carregados</h3>
        <p>• Imóveis: <strong>{len(df_imoveis_demografico):,}</strong></p>
        <p>• Municípios: <strong>{len(df_idade):,}</strong></p>
      </div>
    </div>
  </div>

  <script>
    let isResizing = false;
    let resizeTimeout;

    function toggleSidebar() {{
      const body = document.body;
      const sidebar = document.getElementById('sidebar');
      const overlay = document.getElementById('overlay');
      const content = document.getElementById('content');
      
      body.classList.toggle('sidebar-open');
      sidebar.classList.toggle('open');
      overlay.classList.toggle('active');
      
      content.classList.add('resizing');
      
      clearTimeout(resizeTimeout);
      resizeTimeout = setTimeout(() => {{
        content.classList.remove('resizing');
        resizePlotlyGraphsSmooth();
      }}, 450);
    }}

    function showTab(tabName) {{
      document.querySelectorAll('#sidebar a').forEach(link => {{
        link.classList.remove('active');
      }});
      
      document.querySelectorAll('#content > div').forEach(div => {{
        div.style.display = 'none';
      }});
      
      const targetSection = document.getElementById(tabName);
      targetSection.style.display = 'block';
      
      event.target.closest('a').classList.add('active');
      
      if (window.innerWidth <= 768) {{
        toggleSidebar();
      }}
      
      requestAnimationFrame(() => {{
        resizePlotlyGraphsSmooth();
      }});
    }}

    function resizePlotlyGraphsSmooth() {{
      if (isResizing) return;
      
      isResizing = true;
      
      if (typeof Plotly !== 'undefined') {{
        requestAnimationFrame(() => {{
          const visiblePlots = document.querySelectorAll('.content-section[style*="display:block"] .js-plotly-plot, .content-section[style*="display: block"] .js-plotly-plot');
          
          visiblePlots.forEach(plot => {{
            try {{
              Plotly.Plots.resize(plot);
            }} catch(e) {{
              console.log('Resize skipped:', e);
            }}
          }});
          
          isResizing = false;
        }});
      }} else {{
        isResizing = false;
      }}
    }}

    function toggleTheme() {{
      document.body.classList.toggle('light-theme');
      const themeBtn = document.querySelector('#theme-toggle i');
      const isLight = document.body.classList.contains('light-theme');
      themeBtn.className = isLight ? 'fas fa-sun' : 'fas fa-moon';
      localStorage.setItem('theme', isLight ? 'light' : 'dark');
    }}

    // ⭐ SWITCH DEMOGRÁFICO (CORRIGIDO)
    document.addEventListener('DOMContentLoaded', function() {{
      console.log('🔍 Inicializando dashboard...');
      
      const sliderRaio = document.getElementById('slider-raio');
      if (sliderRaio) {{
        sliderRaio.addEventListener('input', function(e) {{
          document.getElementById('raio-valor').textContent = e.target.value;
        }});
      }}

      const switchDemografico = document.getElementById('switch-demografico');
      const containerFaixas = document.getElementById('container-faixas');

      if (switchDemografico && containerFaixas) {{
        console.log('✓ Switch encontrado!');
        
        switchDemografico.addEventListener('change', function() {{
          console.log('Switch alterado:', this.checked);
          containerFaixas.style.display = this.checked ? 'block' : 'none';
        }});
        
        containerFaixas.style.display = switchDemografico.checked ? 'block' : 'none';
      }} else {{
        console.error('❌ Switch ou container não encontrado');
      }}

      const btnAnalisar = document.getElementById('btn-analisar');
      if (btnAnalisar) {{
        btnAnalisar.addEventListener('click', async function() {{
          const municipio = document.getElementById('select-municipio')?.value;
          const raio = document.getElementById('slider-raio')?.value;
          const usarFiltro = document.getElementById('switch-demografico')?.checked;
          const faixasSelect = document.getElementById('select-faixas');
          const faixas = faixasSelect ? Array.from(faixasSelect.selectedOptions).map(opt => opt.value) : [];

          if (!municipio) {{
            mostrarErro('Por favor, selecione um município.');
            return;
          }}

          if (usarFiltro && faixas.length === 0) {{
            mostrarErro('Por favor, selecione ao menos uma faixa etária.');
            return;
          }}

          document.getElementById('loading-indicator').style.display = 'block';
          document.getElementById('mensagem-inicial').style.display = 'none';
          document.getElementById('mensagem-erro').style.display = 'none';
          document.getElementById('stats-cards').style.display = 'none';
          document.getElementById('mapa-resultado').style.display = 'none';

          try {{
            setTimeout(() => {{
              mostrarErro('Backend não implementado. Execute api_dashboard.py para ativar.');
              document.getElementById('loading-indicator').style.display = 'none';
            }}, 1000);
          }} catch (error) {{
            mostrarErro('Erro: ' + error.message);
          }}
        }});
      }}

      const savedTheme = localStorage.getItem('theme');
      if (savedTheme === 'light') {{
        document.body.classList.add('light-theme');
        document.querySelector('#theme-toggle i').className = 'fas fa-sun';
      }}
      
      setTimeout(() => {{ resizePlotlyGraphsSmooth(); }}, 600);
    }});

    function mostrarErro(mensagem) {{
      document.getElementById('texto-erro').textContent = mensagem;
      document.getElementById('mensagem-erro').style.display = 'flex';
      document.getElementById('mensagem-inicial').style.display = 'none';
      document.getElementById('stats-cards').style.display = 'none';
      document.getElementById('mapa-resultado').style.display = 'none';
    }}

    document.addEventListener('keydown', (e) => {{
      if (e.key === 'Escape' && document.getElementById('sidebar').classList.contains('open')) {{
        toggleSidebar();
      }}
    }});

    let windowResizeTimeout;
    window.addEventListener('resize', () => {{
      clearTimeout(windowResizeTimeout);
      windowResizeTimeout = setTimeout(() => {{ resizePlotlyGraphsSmooth(); }}, 150);
    }});
  </script>
</body>
</html>
"""

with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print("✅ Dashboard gerado com sucesso: dashboard.html")
print("🔍 Abra o arquivo e pressione F12 para ver os logs do console")

✅ Dashboard gerado com sucesso: dashboard.html
🔍 Abra o arquivo e pressione F12 para ver os logs do console
